# 🪰 FlyFlappyBird: Embodied Drosophila MaleCNS Connectome Simulation

**A biophysically grounded closed-loop neural simulation where the complete *Drosophila* male central nervous system connectome (165K neurons, 6.2M synapses from Berg et al. 2026, *Cell*) plays Flappy Bird.**

This notebook runs seamlessly on both **Kaggle** (GPU T4/P100 or CPU) and **Local machines** (Apple Silicon M-Series MPS or CPU).

## 1. Environment & Path Setup
Automatically set the working directory to the project root and ensure all project packages are importable.

In [ ]:
import os
import sys

# Ensure working directory is set to repository root
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Active Working Directory: {os.getcwd()}")

In [ ]:
# Install dependencies (quietly if already satisfied)
!pip install -q -r requirements.txt

## 2. Hardware Acceleration Check
Inspect the compute device: NVIDIA CUDA (Kaggle T4 / P100), Apple MPS (Mac M4), or multi-threaded CPU.

In [ ]:
import torch
import config as CFG

device = CFG.get_device(verbose=True)
print(f"Device selected for sparse SpMV: {device}")
if device.type == "cuda":
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3. neuPrint Authentication & Mode Selection
If you have a Janelia neuPrint API token, you can provide it via Kaggle User Secrets or environment variable.
If no token is supplied, the simulation automatically runs in **Synthetic Connectome Mode (`--mock`)** with zero network dependencies.

In [ ]:
neuprint_token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    neuprint_token = user_secrets.get_secret("NEUPRINT_TOKEN")
    print("[Kaggle] Loaded NEUPRINT_TOKEN from user secrets.")
except Exception:
    neuprint_token = os.environ.get("NEUPRINT_TOKEN", None)

if neuprint_token and neuprint_token != "PASTE_YOUR_TOKEN_HERE":
    os.environ["NEUPRINT_TOKEN"] = neuprint_token
    print("neuPrint token configured. Real connectome download/run enabled.")
else:
    print("No neuPrint token detected. Running in high-speed synthetic connectome mode (--mock).")

## 4. Biological Circuit Architecture
The simulation routes biological signal flow through 6 functional stages:
1. **Sensory Encoding**: Game frames mapped retinotopically to $R1$–$R6$ photoreceptor currents.
2. **Visual Projections**: Lobula visual projection neurons ($LoVP92$, $TmY21$) detect obstacle boundaries.
3. **Central Hubs**: Central brain interneurons integrate inputs with Dale's principle sign constraints.
4. **Descending Commands**: Steering and pursuit descending neurons ($DNg13$, $DNa02$) project to the thoracic ganglion.
5. **Motor Decoding**: Ventral nerve cord wing pre-motor neurons ($TN1A$, $vPR9$, $dPR1$) trigger wing flaps.
6. **Dopaminergic Learning**: $PPL101$ dopamine neurons emit rewards ($+1.0$ pipe, $-5.0$ collision) gating reward-modulated Hebbian plasticity ($R$-STDP) at Kenyon cell synapses.

## 5. Running the Simulation
We invoke `run_simulation()` directly in Python with real-time telemetry and dual-panel dashboard visualization.

In [ ]:
from run import run_simulation

# Run 3 episodes of synthetic connectome with dashboard video recording
results = run_simulation(
    episodes=3,
    mock=True,
    no_viz=False,
    viz_interval=5,
    device=device,
)

print("\n--- Simulation Results ---")
print(f"Episodes Completed: {results['episodes']}")
print(f"Best Score:         {results['best_score']}")
print(f"Video Saved To:     {results['video_path']}")

## 6. Inline Dashboard Visualization
Display the generated simulation video (or animated GIF) directly inside the notebook cell.

In [ ]:
from IPython.display import Video, Image, display

video_file = results.get("video_path")
if video_file and os.path.exists(video_file):
    if video_file.endswith(".mp4"):
        display(Video(video_file, embed=True, width=720))
    elif video_file.endswith(".gif"):
        display(Image(video_file, width=720))
else:
    print("No video file found to display.")

## 7. Full-Scale Connectome Run (165,122 Neurons)
To simulate the real biological brain from the cached dataset (downloaded via `python -m data.fetch_data`):
```python
if os.path.exists(CFG.CONN_NPZ) and os.path.exists(CFG.NEURON_CSV):
    results_real = run_simulation(episodes=2, mock=False, no_viz=True, device=device)
    print("Real connectome run complete:", results_real)
else:
    print("Run 'python -m data.fetch_data' first to cache the full 165K-neuron dataset.")
```

## References
1. **Berg, S., et al.** (2026). *Sexual dimorphism in the complete Drosophila male central nervous system*. **Cell**, 189, 5504–5526. [doi:10.1016/j.cell.2025.10.045](https://doi.org/10.1016/j.cell.2025.10.045)
2. **Lappalainen, J.K., et al.** (2024). *Connectome-constrained networks predict neural activity across the fly visual system*. **Nature**, 634, 1132–1140. [doi:10.1038/s41586-024-07939-3](https://doi.org/10.1038/s41586-024-07939-3)
3. **Aso, Y., et al.** (2014). *The neuronal architecture of the mushroom body provides a logic for associative learning*. **eLife**, 3:e04577. [doi:10.7554/eLife.04577](https://doi.org/10.7554/eLife.04577)
4. **Dayan, P. & Abbott, L.F.** (2001). *Theoretical Neuroscience*. MIT Press. Ch. 5: Model Neurons I — Leaky Integrate-and-Fire.
5. **Wormuth, A.** (2024). *DoomFly: MaleCNS connectome playing Doom*. [GitHub](https://github.com/awormuth/DoomFly)